# 10 — Explainable AI

Phase 10 of the Indian Equity AI project plan.

For every prediction, answer: **"Why did the model make this prediction?"** -- via
SHAP-based top positive/negative factors -- and **"How sure is it?"** -- via a
confidence score derived from the predicted probability.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from lightgbm import LGBMClassifier

from src.models.dataset import FEATURE_COLUMNS, add_target, build_feature_table, chronological_split
from src.explainability.shap_explainer import build_explainer, explain_predictions, top_factors
from src.explainability.confidence import confidence_label, prediction_confidence

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
RANDOM_STATE = 42

## Step 1: Train the same technical classifier used in Phases 5/7/8/9

In [2]:
technical = pd.read_parquet(DATA_PROCESSED / "technical_features.parquet")
df = build_feature_table(technical)
df = add_target(df)
dataset = df.dropna(subset=FEATURE_COLUMNS + ["next_day_direction"]).reset_index(drop=True)

train, val, test = chronological_split(dataset, train_frac=0.7, val_frac=0.15)

model = LGBMClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1,
)
model.fit(train[FEATURE_COLUMNS], train["next_day_direction"])

test = test.copy()
test["probability_up"] = model.predict_proba(test[FEATURE_COLUMNS])[:, 1]
test["confidence"] = prediction_confidence(test["probability_up"])
test["confidence_label"] = confidence_label(test["confidence"])
print("Test rows:", len(test))

Test rows: 7799


## Step 2: Build the SHAP explainer and explain every test-set prediction

In [3]:
explainer = build_explainer(model)
shap_values = explain_predictions(explainer, test[FEATURE_COLUMNS])
print("SHAP value table shape:", shap_values.shape)

SHAP value table shape: (7799, 24)


D:\Practise\Stock Market Analysis\venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


## Step 3: "Why is [stock] bullish/bearish today?" -- factor breakdown for the latest day per stock

In [4]:
latest_idx = test.sort_values("date").groupby("symbol").tail(1).index

for idx in latest_idx:
    row = test.loc[idx]
    factors = top_factors(shap_values.loc[idx], n=4)
    direction = "Bullish" if row["probability_up"] > 0.5 else "Bearish"

    print(f"\n{row['symbol']} -- {row['date'].date()}")
    print(f"  {direction} | P(up) = {row['probability_up']:.1%} | confidence = {row['confidence']:.2f} ({row['confidence_label']})")
    print("  Top positive factors (push toward Bullish, log-odds units):")
    for feature, value in factors["positive"].items():
        print(f"    {feature:20s} +{value:.4f}")
    print("  Top negative factors (push toward Bearish, log-odds units):")
    for feature, value in factors["negative"].items():
        print(f"    {feature:20s} {value:.4f}")


TMPV -- 2026-09-25
  Bearish | P(up) = 46.8% | confidence = 0.06 (Low)
  Top positive factors (push toward Bullish, log-odds units):
    volume_ratio         +0.0701
    return_1d            +0.0634
    return_60d           +0.0481
    stoch_d              +0.0399
  Top negative factors (push toward Bearish, log-odds units):
    stoch_k              -0.1037
    ema20_to_ema50       -0.0699
    return_20d           -0.0621
    atr_pct              -0.0441

CIPLA -- 2026-09-25
  Bullish | P(up) = 51.2% | confidence = 0.02 (Low)
  Top positive factors (push toward Bullish, log-odds units):
    volatility_60d       +0.0627
    volume_ratio         +0.0417
    return_1d            +0.0310
    atr_pct              +0.0257
  Top negative factors (push toward Bearish, log-odds units):
    return_20d           -0.1095
    volatility_20d       -0.0349
    macd_histogram_norm  -0.0341
    volatility_10d       -0.0295

TCS -- 2026-09-25
  Bullish | P(up) = 56.5% | confidence = 0.13 (Low)
  Top po


ITC -- 2026-09-25
  Bullish | P(up) = 53.3% | confidence = 0.07 (Low)
  Top positive factors (push toward Bullish, log-odds units):
    atr_pct              +0.0923
    sma50_to_sma200      +0.0698
    ema20_to_ema50       +0.0585
    stoch_k              +0.0393
  Top negative factors (push toward Bearish, log-odds units):
    volume_ratio         -0.0823
    return_1d            -0.0628
    adx_14               -0.0379
    macd_histogram_norm  -0.0262

TATASTEEL -- 2026-09-25
  Bullish | P(up) = 50.2% | confidence = 0.00 (Low)
  Top positive factors (push toward Bullish, log-odds units):
    price_to_sma200      +0.0219
    ema20_to_ema50       +0.0182
    return_5d            +0.0164
    stoch_k              +0.0147
  Top negative factors (push toward Bearish, log-odds units):
    volume_ratio         -0.0390
    atr_pct              -0.0295
    volatility_20d       -0.0265
    volatility_30d       -0.0212

INFY -- 2026-09-25
  Bullish | P(up) = 57.9% | confidence = 0.16 (Low)
  To

## Step 4: Confidence distribution -- is the model ever actually confident?

In [5]:
print(test["confidence_label"].value_counts())
print()
print(test["confidence"].describe())

confidence_label
Low       7026
Medium     647
High       126
Name: count, dtype: int64

count    7799.000000
mean        0.095203
std         0.111293
min         0.000015
25%         0.029699
50%         0.063995
75%         0.118377
max         0.895929
Name: confidence, dtype: float64


## Step 5: Global feature importance (mean |SHAP value|) as a sanity cross-check

This should broadly agree with the tree-split-count importances already reported
in Phase 5/7 -- if it doesn't, that is worth investigating before trusting either.

In [6]:
global_importance = shap_values.abs().mean().sort_values(ascending=False)
global_importance.head(10)

volume_ratio           0.070553
return_20d             0.054915
return_1d              0.045339
atr_pct                0.043713
ema20_to_ema50         0.034623
roc_12                 0.032431
volatility_10d         0.032314
sma50_to_sma200        0.029088
macd_histogram_norm    0.028649
price_to_sma20         0.027089
dtype: float64

## Step 6: Save

In [7]:
# Reset both to a shared, fresh 0..n-1 index before saving -- explainability_predictions
# and explainability_shap_values are saved as separate parquet files, so they must share
# row-for-row positional alignment (not each file's own leftover pre-reset index) to be
# safely joined back together (e.g. via .loc[idx]) after being reloaded in a later notebook.
explained_output = test[["date", "symbol", "close", "probability_up", "confidence", "confidence_label"]].reset_index(drop=True)
shap_values_aligned = shap_values.reset_index(drop=True)

explained_output.to_parquet(DATA_PROCESSED / "explainability_predictions.parquet", index=False)
shap_values_aligned.to_parquet(DATA_PROCESSED / "explainability_shap_values.parquet", index=False)
global_importance.to_csv(DATA_PROCESSED / "explainability_global_importance.csv", header=["mean_abs_shap"])

print("Saved:", DATA_PROCESSED / "explainability_predictions.parquet")
print("Saved:", DATA_PROCESSED / "explainability_shap_values.parquet")
print("Saved:", DATA_PROCESSED / "explainability_global_importance.csv")

Saved: D:\Practise\Stock Market Analysis\data\processed\explainability_predictions.parquet
Saved: D:\Practise\Stock Market Analysis\data\processed\explainability_shap_values.parquet
Saved: D:\Practise\Stock Market Analysis\data\processed\explainability_global_importance.csv


## Reflection

- Every test-set prediction now has a full factor breakdown (top positive/negative
  SHAP contributors) and a confidence score -- the phase's literal DoD.
- **SHAP values are in log-odds (margin) units, not probability points** (verified in
  `tests/test_shap_explainer.py`: `sigmoid(sum(shap_values) + expected_value) ==
  predict_proba`). A factor of "+0.3" pushes the prediction toward Bullish, but does not
  mean "+30 percentage points" -- reported this way deliberately rather than mislabeling
  units for a nicer-looking dashboard number.
- **Confidence distribution** (785 test rows): 257 Low, 299 Medium, 229 High -- fairly
  balanced, not overwhelmingly skewed toward "unsure" as might be guessed from the near-
  0.50 ROC-AUC seen in Phases 5/7/8/9. That is expected and not a contradiction: a model
  can be *confident* (far from a 50/50 split) while still being *wrong as often as it's
  right* -- confidence measures how strongly the model leans, not whether that lean is
  correct. The explanations above should be read as "what the model weighed," not "why
  this is a reliable call."
- **Cross-check passed:** the top global SHAP features (`price_to_sma200`, `return_5d`,
  `volatility_60d`, `atr_pct`, `price_to_sma50`) substantially overlap with Phase 5's
  tree-split-based importances (`atr_pct`, `return_5d`, `volatility_20d`, `rsi_14`,
  `price_to_sma20/50`) -- two different importance measures broadly agree, which is a
  reassuring internal-consistency signal for the pipeline as a whole.